# 09. Manual Fallback Test

`classification_full` 또는 `classification_detail`에서 기타로 분류된 memo_id를 사람이 검토한 뒤, 기존 topic으로 재배치하거나 그대로 기타로 유지하는 테스트 노트북입니다.

In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.manual_fallback_applier as manual_fallback_applier
import taxonomy.review_decision_writer as review_decision_writer

importlib.reload(config_loader)
importlib.reload(manual_fallback_applier)
importlib.reload(review_decision_writer)

from common.config_loader import load_config, get_output_table
from taxonomy.manual_fallback_applier import (
    load_manual_fallback_candidates,
    build_manual_fallback_decision_df,
    apply_and_save_manual_fallback,
)
from taxonomy.review_decision_writer import save_review_decisions

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")
print("config loaded")

In [ ]:
INPUT_TABLE_KEY = "classification_full"
MODEL_KEY = "gpt_55"
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]

TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-01. 채널/컨텐츠 APP"
TARGET_SC_MEASUREMENT = 1

PROMPT_VERSION = config["version"]["prompt_version"]
TAXONOMY_VERSION = config["version"]["taxonomy_version"]

topic_pool_table = get_output_table(config, "topic_pool")
display(
    spark.table(topic_pool_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == PROMPT_VERSION)
    .where(F.col("taxonomy_version") == TAXONOMY_VERSION)
    .select("topic_order", "topic", "description")
    .orderBy("topic_order")
)

In [ ]:
candidates_df = load_manual_fallback_candidates(
    spark=spark,
    config=config,
    input_table_key=INPUT_TABLE_KEY,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    model_version=MODEL_VERSION_VALUE,
    prompt_version=PROMPT_VERSION,
    taxonomy_version=TAXONOMY_VERSION,
    max_rows=100,
)

print({"candidate_cnt": candidates_df.count()})
display(candidates_df)

## 사람 검토 입력 방식

아래 셀의 `manual_rows`에 검토 결과를 넣습니다.

- 기존 topic으로 보낼 경우: `approved_action = "reassign_existing_topic"`, `approved_topic = "주제명"`
- 그대로 기타로 둘 경우: `approved_action = "keep_others"`, `approved_topic = ""`
- 아무 결정도 하지 않을 메모는 `manual_rows`에 넣지 않습니다.

In [ ]:
# 예시입니다. 위 candidates_df에서 memo_id/sample_memo를 보고 필요한 행만 직접 입력하세요.
# approved_topic은 반드시 위 topic_pool에 존재하는 topic명을 그대로 입력하는 것을 권장합니다.
manual_rows = [
    # {
    #     "memo_id": "붙여넣기",
    #     "approved_action": "reassign_existing_topic",
    #     "approved_topic": "앱 품질/만족",
    #     "reviewer": "jungryo.lee",
    #     "review_comment": "앱에 대한 구체 사유 없는 긍정이므로 기존 topic으로 재배치",
    # },
    # {
    #     "memo_id": "붙여넣기",
    #     "approved_action": "keep_others",
    #     "approved_topic": "",
    #     "reviewer": "jungryo.lee",
    #     "review_comment": "기존 topic에 맞지 않아 기타 유지",
    # },
]

if not manual_rows:
    print("manual_rows is empty. 검토할 행을 입력한 뒤 이 셀을 다시 실행하세요.")
else:
    manual_input_df = spark.createDataFrame(manual_rows)
    reviewed_df = candidates_df.join(manual_input_df, on="memo_id", how="inner")
    display(reviewed_df.select(
        "memo_id",
        "sample_memo",
        "current_pred_topic",
        "approved_action",
        "approved_topic",
        "reviewer",
        "review_comment",
    ))

In [ ]:
if not manual_rows:
    print("skip: manual_rows is empty")
else:
    decision_df = build_manual_fallback_decision_df(
        reviewed_df=reviewed_df,
        config=config,
        source_table_key=INPUT_TABLE_KEY,
        created_by="manual_review",
    )
    display(decision_df.select(
        "decision_id",
        "candidate_type",
        "sample_memo",
        "approved_action",
        "approved_topic",
        "decision_status",
        "reviewer",
        "review_comment",
    ))

    saved_decision_table = save_review_decisions(
        spark=spark,
        config=config,
        decision_df=decision_df,
        write_mode="replace_decisions",
    )
    print("saved_decision_table =", saved_decision_table)

In [ ]:
if not manual_rows:
    print("skip: manual_rows is empty")
else:
    apply_result = apply_and_save_manual_fallback(
        spark=spark,
        config=config,
        input_table_key=INPUT_TABLE_KEY,
        cate_1_depth=TARGET_CATE_1_DEPTH,
        cate_2_depth=TARGET_CATE_2_DEPTH,
        sc_measurement=TARGET_SC_MEASUREMENT,
        model_version=MODEL_VERSION_VALUE,
        prompt_version=PROMPT_VERSION,
        taxonomy_version=TAXONOMY_VERSION,
        write_mode="replace_groups",
    )
    print(apply_result)

In [ ]:
target_table = get_output_table(config, INPUT_TABLE_KEY)
display(
    spark.table(target_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == PROMPT_VERSION)
    .where(F.col("taxonomy_version") == TAXONOMY_VERSION)
    .groupBy("pred_topic", "pred_topic_type", "classification_stage")
    .agg(F.count("*").alias("row_cnt"), F.countDistinct("memo_id").alias("memo_id_cnt"))
    .orderBy(F.col("row_cnt").desc())
)